# Apache Spark avanzado: Optimización y técnicas de performance

**Sesión 25:** Ejecución sobre el clúster (YARN) y Spark UI (parte 1 de 2)

**Dataset:** NYC Taxi Trip Data (Yellow Taxi - 2024)  


## Paso 0: Inicializar SparkSession

Configurar Spark con parámetros optimizados para el clúster.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,           # Para acceder a columnas
    avg,           # Promedios
    when,          # Condicionales
    to_date,       # Conversión a fecha
    concat,        # Concatenación de strings
    lit,           # Valores literales
    broadcast      # Hint de broadcast join
)
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel  # Para niveles de persistencia
import time

# Crear SparkSession con configuración para clúster
spark = SparkSession.builder \
    .master("yarn") \
    .appName("NYC Taxi Analysis") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

# Verificar configuración
print("Spark Version:", spark.version)
print("Master:", spark.sparkContext.master)
print("App ID:", spark.sparkContext.applicationId)

Spark Version: 3.5.7
Master: yarn
App ID: application_1790631121365_0001


---

### ⚠️ ADVERTENCIA IMPORTANTE: Error Común con `import *`

#### El problema

Si se visualiza este error en Python:
```
❌ Error: min() takes 1 positional argument but 2 were given
❌ Error: sum() takes 1 positional argument but 2 were given
```

**Causa:** `from pyspark.sql.functions import *`

#### ¿Por qué se genera?

Cuando se usa `import *`, se importan **TODAS** las funciones de PySpark SQL, incluyendo:
- `min()`, `max()`, `sum()` → diseñadas para **columnas de DataFrames**

Estas **sobrescriben** las funciones built-in de Python:
- `min(a, b)` de Python → Reemplazada por `min(col)` de PySpark
- `sum([1,2,3])` de Python → Reemplazada por `sum(col)` de PySpark

#### Solución: Importaciones explícitas

**MALO:**
```python
from pyspark.sql.functions import *  # Se aconseja evitar SIEMPRE
```

**BUENO:**
```python
from pyspark.sql.functions import col, avg, sum as spark_sum, count, when
```
#### Consideración:

> **Si es posible listar lo que se necesita, NUNCA usar `import *`**

### Si ya se ha generado el error

**Opción 1 (recomendado):** Cambiar importaciones
```python
from pyspark.sql.functions import col, avg, count  # Solo lo necesario
```

**Opción 2:** Usar built-in explícitamente
```python
import builtins
percent = builtins.min(100, downloaded/total * 100)
```

---

### Configuración de Spark en el clúster de Hadoop

#### Arquitectura del clúster

Se cuenta con 3 máquinas virtuales:
- **1 Nodo Maestro**: 8GB RAM, 4 CPUs
- **2 Nodos Workers**: 8GB RAM cada uno, 4 CPUs cada uno

> ⚠️ **IMPORTANTE**: Los recursos de cada nodo son **INDEPENDIENTES**.  
> No se suman entre sí. Cada máquina gestiona su propia memoria.

---

#### Distribución de roles

##### Nodo maestro (donde estamos ejecutando Jupyter):
- Driver de Spark (coordina todo el trabajo)
- NameNode de HDFS (gestiona metadatos de archivos)
- ResourceManager de YARN (asigna recursos del clúster)
- Este notebook de Jupyter

##### Nodos Workers (ejecutan el trabajo real):
- Executors de Spark (procesan los datos)
- DataNodes de HDFS (almacenan bloques de datos)
- NodeManagers de YARN (gestionan recursos locales)

---

#### Explicación de la configuración

##### 1. `.master("yarn")`
Se usa YARN como gestor de recursos del clúster.

##### 2. `.config("spark.executor.memory", "4g")`
- Cada executor usa **4GB** de los 8GB disponibles en su worker
- Se deja aproximadamente 4GB para:
  - Sistema Operativo: 1.5GB
  - YARN (NodeManager): 0.5GB
  - HDFS (DataNode): 0.5GB
  - Overhead de JVM: 0.4GB
  - Margen de seguridad: 1.1GB

##### 3. `.config("spark.executor.cores", "2")`
- Cada executor usa **2 de los 4 CPUs** disponibles en su worker
- Se deja 2 CPUs para otros procesos del sistema

##### 4. `.config("spark.executor.instances", "2")`
- Se crean **2 executors en total** (1 por cada worker)
- **Total de paralelismo**: 2 executors × 2 cores = **4 tareas simultáneas**

##### 5. `.config("spark.sql.shuffle.partitions", "8")`
- Datos se dividen en **8 particiones** durante operaciones pesadas (joins, groupBy, etc.)
- **Cálculo**: 8 particiones ÷ 4 cores = 2 particiones por core (ideal)
- El default es 200, **demasiado para un clúster pequeño**

##### 6. `.config("spark.sql.adaptive.enabled", "true")`
- Spark optimiza el plan de ejecución **DURANTE** la ejecución
- Ajusta particiones, joins y shuffles automáticamente basándose en datos reales

##### 7. `.config("spark.sql.adaptive.coalescePartitions.enabled", "true")`
- Combina particiones pequeñas automáticamente
- Evita overhead de manejar muchas particiones con pocos datos

---

#### ¿Por qué esta configuración es óptima?

| Aspecto | Beneficio |
|---------|-----------|
| **Memoria conservadora** | Se usa 50% de RAM por seguridad, se evita swapping |
| **CPUs balanceados** | Se dejan recursos para SO y servicios de Hadoop |
| **Particiones ajustadas** | 8 particiones balanceadas con el paralelismo real del clúster (4 cores) |
| **Optimización adaptativa** | AQE mejora el rendimiento sin intervención manual |

---

#### Recordatorio importante sobre el Driver

Cuando se ejecuta desde Jupyter, el **Driver corre en el NODO MAESTRO** (no en los workers).

**El Driver:**
- Coordina las tareas que ejecutarán los executors
- Mantiene el contexto de Spark (SparkSession)
- Recibe resultados de los executors
- Puede usar aproximadamente 2-4GB del maestro

Por eso el nodo maestro debe tener recursos suficientes **incluso si no ejecuta tareas de procesamiento de datos**.

---

---

## Sección 1: Descarga y carga en HDFS

**Estrategia de descarga:**
1. Descargar a `/tmp/` (almacenamiento local temporal)
2. Subir a HDFS usando comandos nativos y eliminar el archivo local al finalizar cada mes
3. Verificar la carga exitosa

### Paso 1.1: Descargar el dataset

In [2]:
import urllib.request
import os

# Configuración para descargar todo el año 2024
months = [f"2024-{str(i).zfill(2)}" for i in range(1, 13)]  # 2024-01 hasta 2024-12

print(f"{'='*70}")
print(f" Descargando {len(months)} meses de datos NYC Taxi")
print(f"{'='*70}\n")

for idx, month in enumerate(months, 1):
    dataset_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{month}.parquet"
    local_path = f"/tmp/yellow_tripdata_{month}.parquet"
    
    print(f"[{idx}/{len(months)}] Descargando {month}...")
    print(f"   URL: {dataset_url}")
    
    try:
        urllib.request.urlretrieve(dataset_url, local_path)
        file_size = os.path.getsize(local_path) / (1024 * 1024)
        print(f"Completado: {file_size:.2f} MB\n")
    except Exception as e:
        print(f"   ❌ Error: {str(e)}\n")
        continue

print(f"{'='*70}")
print("Descargas completadas")
print(f"{'='*70}\n")

 Descargando 12 meses de datos NYC Taxi

[1/12] Descargando 2024-01...
   URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet
Completado: 47.65 MB

[2/12] Descargando 2024-02...
   URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet
Completado: 48.02 MB

[3/12] Descargando 2024-03...
   URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet
Completado: 57.30 MB

[4/12] Descargando 2024-04...
   URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-04.parquet
Completado: 56.39 MB

[5/12] Descargando 2024-05...
   URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-05.parquet
Completado: 59.66 MB

[6/12] Descargando 2024-06...
   URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-06.parquet
Completado: 57.09 MB

[7/12] Descargando 2024-07...
   URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-07.parquet
C

### Paso 1.2: Subir a HDFS

In [3]:
import subprocess

# Rutas HDFS
hdfs_base_path = "hdfs://hadoop-virtualbox:9000/datasets"

print(f"{'='*70}")
print(" Subiendo archivos a HDFS")
print(f"{'='*70}\n")

# Crear directorio en HDFS si no existe
try:
    subprocess.run(["hdfs", "dfs", "-mkdir", "-p", hdfs_base_path], 
                   check=True, capture_output=True)
    print(f"Directorio HDFS listo: {hdfs_base_path}\n")
except subprocess.CalledProcessError as e:
    print(f"⚠️ Advertencia: {e.stderr.decode()}\n")

# Subir todos los archivos a HDFS
for idx, month in enumerate(months, 1):
    local_path = f"/tmp/yellow_tripdata_{month}.parquet"
    hdfs_file_path = f"{hdfs_base_path}/yellow_tripdata_{month}.parquet"
    
    # Verificar que el archivo local existe
    if not os.path.exists(local_path):
        print(f"[{idx}/{len(months)}] ⚠️ Saltando {month} (no encontrado localmente)")
        continue
    
    print(f"[{idx}/{len(months)}] Subiendo {month} a HDFS...")
    
    try:
        subprocess.run(
            ["hdfs", "dfs", "-put", "-f", local_path, hdfs_file_path],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"Subido")
        
        # Opcional: Eliminar archivo local para liberar espacio
        os.remove(local_path)
        print(f"Archivo local eliminado\n")
        
    except subprocess.CalledProcessError as e:
        print(f"   ❌ Error: {e.stderr}\n")

print(f"{'='*70}")
print("Proceso de carga a HDFS completado")
print(f"{'='*70}\n")

 Subiendo archivos a HDFS

Directorio HDFS listo: hdfs://hadoop-virtualbox:9000/datasets

[1/12] Subiendo 2024-01 a HDFS...
Subido
Archivo local eliminado

[2/12] Subiendo 2024-02 a HDFS...
Subido
Archivo local eliminado

[3/12] Subiendo 2024-03 a HDFS...
Subido
Archivo local eliminado

[4/12] Subiendo 2024-04 a HDFS...
Subido
Archivo local eliminado

[5/12] Subiendo 2024-05 a HDFS...
Subido
Archivo local eliminado

[6/12] Subiendo 2024-06 a HDFS...
Subido
Archivo local eliminado

[7/12] Subiendo 2024-07 a HDFS...
Subido
Archivo local eliminado

[8/12] Subiendo 2024-08 a HDFS...
Subido
Archivo local eliminado

[9/12] Subiendo 2024-09 a HDFS...
Subido
Archivo local eliminado

[10/12] Subiendo 2024-10 a HDFS...
Subido
Archivo local eliminado

[11/12] Subiendo 2024-11 a HDFS...
Subido
Archivo local eliminado

[12/12] Subiendo 2024-12 a HDFS...
Subido
Archivo local eliminado

Proceso de carga a HDFS completado



### Paso 1.3: Verificar Carga en HDFS

In [4]:
print("Verificando archivos en HDFS...\n")

# Listar contenido del directorio
result = subprocess.run(
    ["hdfs", "dfs", "-ls", "-h", hdfs_base_path],
    capture_output=True,
    text=True
)

print(result.stdout)

# Verificar información específica del archivo
print("Detalles del archivo subido:\n")
# Nota: hdfs_file_path contiene la ruta del último archivo del bucle (2024-12)
result = subprocess.run(
    ["hdfs", "dfs", "-stat", "%n - %b bytes - Replicación: %r", hdfs_file_path],
    capture_output=True,
    text=True
)
print(result.stdout)

print("\nVerificación completada. Los archivos están disponibles en HDFS.")

Verificando archivos en HDFS...

Found 14 items
-rw-r--r--   2 hadoop supergroup     11.4 M 2026-09-23 16:46 hdfs://hadoop-virtualbox:9000/datasets/global_power_plant_database.csv
drwxr-xr-x   - hadoop supergroup          0 2026-09-23 16:54 hdfs://hadoop-virtualbox:9000/datasets/plantas_parquet
-rw-r--r--   2 hadoop supergroup     47.6 M 2026-09-28 17:02 hdfs://hadoop-virtualbox:9000/datasets/yellow_tripdata_2024-01.parquet
-rw-r--r--   2 hadoop supergroup     48.0 M 2026-09-28 17:02 hdfs://hadoop-virtualbox:9000/datasets/yellow_tripdata_2024-02.parquet
-rw-r--r--   2 hadoop supergroup     57.3 M 2026-09-28 17:02 hdfs://hadoop-virtualbox:9000/datasets/yellow_tripdata_2024-03.parquet
-rw-r--r--   2 hadoop supergroup     56.4 M 2026-09-28 17:02 hdfs://hadoop-virtualbox:9000/datasets/yellow_tripdata_2024-04.parquet
-rw-r--r--   2 hadoop supergroup     59.7 M 2026-09-28 17:02 hdfs://hadoop-virtualbox:9000/datasets/yellow_tripdata_2024-05.parquet
-rw-r--r--   2 hadoop supergroup     57.1 M 

### Paso 1.4: Cargar dataset en Spark desde HDFS

In [5]:
print("Cargando dataset en Spark desde HDFS...\n")

# Medir tiempo de carga
start_time = time.time()

# Cargar desde HDFS
# Leer TODOS los archivos de 2024 con wildcard
df_taxi = spark.read.parquet("hdfs://hadoop-virtualbox:9000/datasets/yellow_tripdata_2024-*.parquet")

# Verificar cuántos datos se cargan
print(f"{'='*70}")
print(f"Dataset NYC Taxi 2024 completo cargado")
print(f"{'='*70}")
print(f"Particiones: {df_taxi.rdd.getNumPartitions()}")
print(f"{'='*70}\n")

# Ejecutar una acción para medir tiempo real de carga
row_count = df_taxi.count()

load_time = time.time() - start_time

print(f"Dataset cargado")
print(f"Tiempo de carga: {load_time:.2f} segundos")
print(f"Número de registros: {row_count:,}")
print(f"Número de columnas: {len(df_taxi.columns)}")

# Mostrar schema
print("\n Schema del DataFrame:\n")
df_taxi.printSchema()

Cargando dataset en Spark desde HDFS...



Dataset NYC Taxi 2024 completo cargado
Particiones: 6



[Stage 1:=======================================>                   (4 + 2) / 6]

Dataset cargado
Tiempo de carga: 7.94 segundos
Número de registros: 41,169,720
Número de columnas: 19

 Schema del DataFrame:

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

### Ejercicio 1: Exploración básica

**Instrucciones:** Completar las siguientes tareas para familiarizarse con el dataset.

1. Mostrar las primeras 5 filas del DataFrame
2. Obtener estadísticas descriptivas de la columna `trip_distance`
3. Contar cuántos viajes hay por cada tipo de pago (`payment_type`)

In [6]:
# 1. Mostrar primeras 5 filas
print(" Primeras 5 filas:\n")
df_taxi.show(5, truncate=False)

# 2. Estadísticas de trip_distance
print("\n Estadísticas de trip_distance:\n")
df_taxi.select("trip_distance").describe().show()

# 3. Conteo por payment_type
print("\n Distribución de tipos de pago:\n")
df_taxi.groupBy("payment_type").count().orderBy("count", ascending=False).show()

 Primeras 5 filas:

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-10-01 00:30:44 |2024-10-01 00:48:26  |1              |3.0          |1         |N                 |162         |246         |1           |18.4       |1.0

+-------+-----------------+
|summary|    trip_distance|
+-------+-----------------+
|  count|         41169720|
|   mean|4.976100785237861|
| stddev|419.2304973495382|
|    min|              0.0|
|    max|        398608.62|
+-------+-----------------+


 Distribución de tipos de pago:



[Stage 8:=======================================>                   (4 + 2) / 6]

+------------+--------+
|payment_type|   count|
+------------+--------+
|           1|30452159|
|           2| 5540088|
|           0| 4091232|
|           4|  794494|
|           3|  291743|
|           5|       4|
+------------+--------+



---

## Sección 2: Análisis de Apache Parquet

### Características principales

**1. Formato columnar:**
```
CSV (orientado a filas):          Parquet (orientado a columnas):
Row1: A, B, C, D                  Col A: [val1, val2, val3, ...]
Row2: A, B, C, D                  Col B: [val1, val2, val3, ...]
Row3: A, B, C, D                  Col C: [val1, val2, val3, ...]
```

**Ventajas:**
- **Mejor compresión:** Datos similares se almacenan juntos
- **Lectura selectiva:** Solo lee columnas necesarias (projection pushdown)
- **Tipos de datos nativos:** Preserva tipos (int, float, timestamp)
- **Metadata incorporada:** Schema, estadísticas, encoding

**2. Compresión eficiente:**
- Snappy (rápido, compresión moderada)
- GZIP (mejor compresión, más lento)
- LZ4, ZSTD (balance)

**3. Predicate pushdown:**
- Filtros se aplican antes de leer los datos
- Usa estadísticas de min/max por **row group** (grupo de filas)
- Salta bloques que no cumplen condiciones

### Paso 2.1: Inspeccionar schema detallado

In [7]:
print("Schema detallado del archivo parquet:\n")
print("=" * 80)

for field in df_taxi.schema.fields:
    print(f" {field.name}")
    print(f"   Tipo: {field.dataType}")
    print(f"   Nullable: {field.nullable}")
    print()

print("=" * 80)
print(f"\n Total de columnas: {len(df_taxi.schema.fields)}")

Schema detallado del archivo parquet:

 VendorID
   Tipo: IntegerType()
   Nullable: True

 tpep_pickup_datetime
   Tipo: TimestampNTZType()
   Nullable: True

 tpep_dropoff_datetime
   Tipo: TimestampNTZType()
   Nullable: True

 passenger_count
   Tipo: LongType()
   Nullable: True

 trip_distance
   Tipo: DoubleType()
   Nullable: True

 RatecodeID
   Tipo: LongType()
   Nullable: True

 store_and_fwd_flag
   Tipo: StringType()
   Nullable: True

 PULocationID
   Tipo: IntegerType()
   Nullable: True

 DOLocationID
   Tipo: IntegerType()
   Nullable: True

 payment_type
   Tipo: LongType()
   Nullable: True

 fare_amount
   Tipo: DoubleType()
   Nullable: True

 extra
   Tipo: DoubleType()
   Nullable: True

 mta_tax
   Tipo: DoubleType()
   Nullable: True

 tip_amount
   Tipo: DoubleType()
   Nullable: True

 tolls_amount
   Tipo: DoubleType()
   Nullable: True

 improvement_surcharge
   Tipo: DoubleType()
   Nullable: True

 total_amount
   Tipo: DoubleType()
   Nullable: True

 c

### Paso 2.2: Comparar parquet vs csv

Se va a escribir el dataset en formato CSV y comparar tamaños y tiempos de lectura.

In [8]:
# Crear una muestra del dataset para comparación
sample_size = 100000  # 100.000 registros para comparación
df_sample = df_taxi.limit(sample_size)

# Rutas para guardar
parquet_test_path = f"{hdfs_base_path}/test_parquet"
csv_test_path = f"{hdfs_base_path}/test_csv"

print(f" Guardando muestra de {sample_size:,} registros en ambos formatos...\n")

# Guardar en Parquet
print("Escribiendo Parquet...")
start_time = time.time()
df_sample.write.mode("overwrite").parquet(parquet_test_path)
parquet_write_time = time.time() - start_time
print(f"   Tiempo de escritura: {parquet_write_time:.2f}s")

# Guardar en CSV
print("\n Escribiendo CSV...")
start_time = time.time()
df_sample.write.mode("overwrite").option("header", "true").csv(csv_test_path)
csv_write_time = time.time() - start_time
print(f"   Tiempo de escritura: {csv_write_time:.2f}s")

 Guardando muestra de 100,000 registros en ambos formatos...

Escribiendo Parquet...


   Tiempo de escritura: 2.49s

 Escribiendo CSV...


[Stage 16:>                                                         (0 + 1) / 1]

   Tiempo de escritura: 1.80s


In [9]:
# Obtener tamaños de archivos en HDFS
import subprocess

def get_hdfs_size(path):
    """Obtiene el tamaño de un directorio en HDFS"""
    result = subprocess.run(
        ["hdfs", "dfs", "-du", "-s", "-h", path],
        capture_output=True,
        text=True
    )
    # Parsear output: "size unit hdfs_path"
    parts = result.stdout.split()
    return f"{parts[0]} {parts[1]}" if len(parts) >= 2 else parts[0]

print("\n Comparación de Tamaños:\n")
print("=" * 60)

parquet_size = get_hdfs_size(parquet_test_path)
csv_size = get_hdfs_size(csv_test_path)

print(f" Parquet: {parquet_size}")
print(f" CSV:     {csv_size}")
print("=" * 60)

print("\n Observaciones:")
print("   • Parquet típicamente usa 50 a 80% menos espacio que CSV")
print("   • La compresión columnar es más eficiente para datos repetitivos")
print("   • Parquet incluye metadata pero aún así es más pequeño")


 Comparación de Tamaños:

 Parquet: 1.9 M
 CSV:     10.5 M

 Observaciones:
   • Parquet típicamente usa 50 a 80% menos espacio que CSV
   • La compresión columnar es más eficiente para datos repetitivos
   • Parquet incluye metadata pero aún así es más pequeño


### Paso 2.3: Comparar tiempos de lectura

In [10]:
print(" Comparando tiempos de lectura...\n")

# Leer Parquet
print("Leyendo Parquet...")
start_time = time.time()
df_parquet_test = spark.read.parquet(parquet_test_path)
count_parquet = df_parquet_test.count()
parquet_read_time = time.time() - start_time
print(f"   Tiempo: {parquet_read_time:.2f}s")
print(f"   Registros: {count_parquet:,}")

# Leer CSV
print("\n Leyendo CSV...")
start_time = time.time()
df_csv_test = spark.read.option("header", "true").option("inferSchema", "true").csv(csv_test_path)
count_csv = df_csv_test.count()
csv_read_time = time.time() - start_time
print(f"   Tiempo: {csv_read_time:.2f}s")
print(f"   Registros: {count_csv:,}")

# Comparación
print("\n" + "=" * 60)
print(" RESULTADOS DE LA COMPARACIÓN")
print("=" * 60)
speedup = csv_read_time / parquet_read_time
print(f"\n Parquet es {speedup:.2f} veces más rápido que CSV para lectura")
print(f"\n Razones principales:")
print(f"   1. No necesita inferir schema (ya está en metadata)")
print(f"   2. Lectura columnar más eficiente")
print(f"   3. Compresión nativa reduce I/O")
print(f"   4. Predicate pushdown permite filtrado temprano")

 Comparando tiempos de lectura...

Leyendo Parquet...
   Tiempo: 0.55s
   Registros: 100,000

 Leyendo CSV...


   Tiempo: 2.95s
   Registros: 100,000

 RESULTADOS DE LA COMPARACIÓN

 Parquet es 5.40 veces más rápido que CSV para lectura

 Razones principales:
   1. No necesita inferir schema (ya está en metadata)
   2. Lectura columnar más eficiente
   3. Compresión nativa reduce I/O
   4. Predicate pushdown permite filtrado temprano


### Paso 2.4: Demostrar projection pushdown

Se va a demostrar cómo Parquet solo lee las columnas necesarias.

In [11]:
print("Demostrando projection pushdown (lectura selectiva de columnas)\n")

# Usar el dataset completo
print("Dataset completo tiene", len(df_taxi.columns), "columnas\n")

# Test 1: Leer todas las columnas
print("Test 1: Leer TODAS las columnas")
start_time = time.time()
result_all = df_taxi.select("*").limit(10000).count()
time_all_cols = time.time() - start_time
print(f"   Tiempo: {time_all_cols:.3f}s\n")

# Test 2: Leer solo 3 columnas
print("Test 2: Leer solo 3 columnas (fare_amount, trip_distance, passenger_count)")
start_time = time.time()
result_3cols = df_taxi.select("fare_amount", "trip_distance", "passenger_count").limit(10000).count()
time_3_cols = time.time() - start_time
print(f"   Tiempo: {time_3_cols:.3f}s\n")

# Comparación
print("=" * 60)
speedup = time_all_cols / time_3_cols
print(f"Lectura selectiva es {speedup:.2f} veces más rápida")
if speedup < 1:
    print(f"\n  Nota: Con .limit(10000) el overhead de planificación puede dominar.")
    print(f"   El beneficio del projection pushdown es apreciable al escanear")
    print(f"   millones de filas sin límite artificiales.")
print(f"\n Esto es projection pushdown:")
print(f"   • Parquet solo lee las columnas solicitadas del disco")
print(f"   • CSV debe leer toda la fila siempre")
print(f"   • Esto se traduce en menos I/O y más velocidad")

Demostrando projection pushdown (lectura selectiva de columnas)

Dataset completo tiene 19 columnas

Test 1: Leer TODAS las columnas
   Tiempo: 0.446s

Test 2: Leer solo 3 columnas (fare_amount, trip_distance, passenger_count)
   Tiempo: 0.365s

Lectura selectiva es 1.22 veces más rápida

 Esto es projection pushdown:
   • Parquet solo lee las columnas solicitadas del disco
   • CSV debe leer toda la fila siempre
   • Esto se traduce en menos I/O y más velocidad


### Ejercicio 2: Análisis de ventajas de Parquet

**Pregunta:** ¿En qué escenarios Parquet NO sería la mejor opción?

**Investigar:**
1. ¿Qué pasa si se necesita agregar una columna a todos los archivos?
2. ¿Es bueno para datos que cambian frecuentemente?
3. ¿Funciona bien para lectura fila por fila (row-by-row)?

In [ ]:
# Escribir sus observaciones aquí (como comentarios)



---

## Sección 3: Cache y persistencia

### Cache y persistencia en Spark

#### ¿Qué es Cache?

Por defecto, Spark **recomputa** transformaciones cada vez que se ejecuta una acción:

```python
df_filtered = df.filter(...)  # Transformación (lazy)
df_filtered.count()           # Acción 1: lee desde origen
df_filtered.show()            # Acción 2: vuelve a leer desde origen
```

Con **cache**, los datos se guardan en memoria después de la primera acción:

```python
df_filtered = df.filter(...).cache()
df_filtered.count()           # Acción 1: lee y guarda en memoria
df_filtered.show()            # Acción 2: lee desde memoria (¡rápido!)
```

### Niveles de persistencia

| Nivel | Memoria | Disco | Serializado | Descripción |
|-------|---------|-------|-------------|-------------|
| `MEMORY_ONLY` | Si | No | Si | Solo RAM, serializado; puede perder particiones si no caben |
| `MEMORY_AND_DISK` | Si | Si | Si | RAM serializado + spill a disco; balance entre velocidad y confiabilidad |
| `MEMORY_AND_DISK_DESER` | Si | Si | No | Default de .cache(); objetos deserializados en RAM, acceso más rápido pero mayor uso de memoria |
| `DISK_ONLY` | No | Si | Si | Solo disco, máximo ahorro de memoria pero lento |

**`.cache()` es equivalente a `.persist(MEMORY_AND_DISK_DESER)`** (objetos deserializados en memoria + spill a disco)

### Cuándo SÍ usar Cache:
1. El DataFrame se usa múltiples veces
2. Las transformaciones son costosas (joins, aggregations)
3. Los datos caben en memoria del clúster
4. Análisis exploratorio iterativo

### Cuándo NO usar Cache:
1. Solo se usa el DataFrame una vez
2. Los datos son muy grandes (no caben en RAM)
3. Las transformaciones son muy simples
4. Pipeline lineal sin reutilización

### ⚠️ ADVERTENCIA: Compatibilidad de StorageLevels

Los niveles de almacenamiento disponibles **varían según la versión de PySpark**.

**Antes de ejecutar las siguientes celdas**, verificar los StorageLevels disponibles en la instalación:

**Nota:** Si se visualiza un error `AttributeError: type object 'StorageLevel' has no attribute...`:
1. Ejecutar el siguiente código para ver los niveles disponibles
2. Ajustar el código con los nombres correctos de la versión

In [12]:
from pyspark import StorageLevel
print([attr for attr in dir(StorageLevel) if not attr.startswith('_')])

['DISK_ONLY', 'DISK_ONLY_2', 'DISK_ONLY_3', 'MEMORY_AND_DISK', 'MEMORY_AND_DISK_2', 'MEMORY_AND_DISK_DESER', 'MEMORY_ONLY', 'MEMORY_ONLY_2', 'NONE', 'OFF_HEAP']


### Paso 3.1: Comparar sin Cache vs con Cache

In [13]:
print("EXPERIMENTO: Medir impacto del cache\n")
print("=" * 70)

# Crear un DataFrame con transformaciones costosas
df_expensive = df_taxi \
    .filter(col("fare_amount") > 10) \
    .filter(col("trip_distance") > 1) \
    .withColumn("fare_per_mile", col("fare_amount") / col("trip_distance")) \
    .withColumn("is_expensive", col("fare_per_mile") > 5)

print("\n DataFrame con transformaciones aplicadas (sin cache):")
print("   • Filtro: fare_amount > 10")
print("   • Filtro: trip_distance > 1")
print("   • Nueva columna: fare_per_mile")
print("   • Nueva columna: is_expensive")
print("\n⚠️  Sin cache, estas transformaciones se recomputarán en CADA acción")

EXPERIMENTO: Medir impacto del cache


 DataFrame con transformaciones aplicadas (sin cache):
   • Filtro: fare_amount > 10
   • Filtro: trip_distance > 1
   • Nueva columna: fare_per_mile
   • Nueva columna: is_expensive

⚠️  Sin cache, estas transformaciones se recomputarán en CADA acción


In [14]:
# Escenario 1: Sin Cache (3 acciones repetidas)
print("\n" + "=" * 70)
print("Escenario 1: Sin cache")
print("=" * 70)

# Acción 1: Contar registros
print("\nAcción 1: Contando registros...")
start = time.time()
count_1 = df_expensive.count()
time_1 = time.time() - start
print(f"   Tiempo: {time_1:.2f}s")
print(f"   Resultado: {count_1:,} registros")

# Acción 2: Calcular promedio
print("\nAcción 2: Calculando promedio de fare_per_mile...")
start = time.time()
avg_fare = df_expensive.agg(avg("fare_per_mile")).collect()[0][0]
time_2 = time.time() - start
print(f"   Tiempo: {time_2:.2f}s")
print(f"   Resultado: ${avg_fare:.2f} por milla")

# Acción 3: Contar viajes caros
print("\nAcción 3: Contando viajes caros...")
start = time.time()
expensive_count = df_expensive.filter(col("is_expensive")).count()
time_3 = time.time() - start
print(f"   Tiempo: {time_3:.2f}s")
print(f"   Resultado: {expensive_count:,} viajes caros")

total_time_no_cache = time_1 + time_2 + time_3
print(f"\n  TIEMPO TOTAL SIN CACHE: {total_time_no_cache:.2f}s")
print("\n Observar que cada acción vuelve a leer desde HDFS y recomputar transformaciones")


Escenario 1: Sin cache

Acción 1: Contando registros...


   Tiempo: 1.20s
   Resultado: 26,193,265 registros

Acción 2: Calculando promedio de fare_per_mile...


   Tiempo: 1.19s
   Resultado: $6.96 por milla

Acción 3: Contando viajes caros...


[Stage 38:======================================>                   (4 + 2) / 6]

   Tiempo: 1.15s
   Resultado: 20,062,995 viajes caros

  TIEMPO TOTAL SIN CACHE: 3.53s

 Observar que cada acción vuelve a leer desde HDFS y recomputar transformaciones


In [15]:
# Escenario 2: Con Cache (mismas 3 acciones)
print("\n" + "=" * 70)
print("Escenario 2: Con cache")
print("=" * 70)

# Aplicar cache
df_cached = df_expensive.cache()
print("\nCache activado con .cache()")
print("   (Equivalente a .persist(StorageLevel.MEMORY_AND_DISK_DESER))")

# Acción 1: Contar registros (materializa el cache)
print("\nAcción 1: Contando registros (MATERIALIZANDO CACHE)...")
start = time.time()
count_1_cached = df_cached.count()
time_1_cached = time.time() - start
print(f"   Tiempo: {time_1_cached:.2f}s")
print(f"   Resultado: {count_1_cached:,} registros")
print("    Datos ahora están en memoria")

# Acción 2: Calcular promedio (usa cache)
print("\nAcción 2: Calculando promedio de fare_per_mile (DESDE CACHE)...")
start = time.time()
avg_fare_cached = df_cached.agg(avg("fare_per_mile")).collect()[0][0]
time_2_cached = time.time() - start
print(f"   Tiempo: {time_2_cached:.2f}s")
print(f"   Resultado: ${avg_fare_cached:.2f} por milla")
print(f"   Speedup: {time_2/time_2_cached:.2f}veces más rápido")

# Acción 3: Contar viajes caros (usa cache)
print("\nAcción 3: Contando viajes caros (DESDE CACHE)...")
start = time.time()
expensive_count_cached = df_cached.filter(col("is_expensive")).count()
time_3_cached = time.time() - start
print(f"   Tiempo: {time_3_cached:.2f}s")
print(f"   Resultado: {expensive_count_cached:,} viajes caros")
print(f"   Speedup: {time_3/time_3_cached:.2f} veces más rápido")

total_time_cached = time_1_cached + time_2_cached + time_3_cached
print(f"\n  TIEMPO TOTAL CON CACHE: {total_time_cached:.2f}s")
overall_speedup = total_time_no_cache / total_time_cached
if overall_speedup >= 1:
    print(f"\n MEJORA GENERAL: {overall_speedup:.2f} veces más rápido con cache")
else:
    print(f"\n MEJORA GENERAL: con cache fue {1/overall_speedup:.2f} veces más LENTO en este escenario (ver interpretación abajo)")


Escenario 2: Con cache

Cache activado con .cache()
   (Equivalente a .persist(StorageLevel.MEMORY_AND_DISK_DESER))

Acción 1: Contando registros (MATERIALIZANDO CACHE)...


   Tiempo: 23.69s
   Resultado: 26,193,265 registros
    Datos ahora están en memoria

Acción 2: Calculando promedio de fare_per_mile (DESDE CACHE)...
   Tiempo: 0.55s
   Resultado: $6.96 por milla
   Speedup: 2.14veces más rápido

Acción 3: Contando viajes caros (DESDE CACHE)...
   Tiempo: 0.57s
   Resultado: 20,062,995 viajes caros
   Speedup: 2.02 veces más rápido

  TIEMPO TOTAL CON CACHE: 24.82s

 MEJORA GENERAL: con cache fue 7.02 veces más LENTO en este escenario (ver interpretación abajo)


> **Interpretación del resultado:** El tiempo total con cache puede ser mayor al tiempo sin cache cuando se miden pocas acciones, ya que la primera acción incluye el costo de **materializar** (leer y almacenar) el DataFrame completo. El beneficio del cache se hace evidente cuando el mismo DataFrame se reutiliza múltiples veces: las acciones 2 y 3 individuales son significativamente más rápidas. En un flujo de análisis exploratorio con 10 o más acciones repetidas, el ahorro acumulado supera el costo inicial de materialización.

### Paso 3.2: Visualizar Cache en Spark UI

**Instrucciones para el estudiante:**

1. Acceder al portal de Spark a través de YARN
2. Navegar a la pestaña **"Storage"**
3. Debería visualizarse el DataFrame cacheado

**Qué observar:**
- **Size in Memory:** Cuánta RAM está usando
- **Size on Disk:** Si hubo spill a disco
- **Fraction Cached:** % del DataFrame en memoria (100% es ideal)
- **Cached Partitions:** Cómo está distribuido

In [16]:
print("\n Si 'Fraction Cached' es inferior al 100%, significa que parte del DataFrame")
print("   se desalojó (evicted) de memoria por falta de espacio.")

# Mostrar información programática
print("\n Información del DataFrame Cacheado:")
print(f"   • Particiones: {df_cached.rdd.getNumPartitions()}")
print(f"   • Storage Level: {df_cached.storageLevel}")


 Si 'Fraction Cached' es inferior al 100%, significa que parte del DataFrame
   se desalojó (evicted) de memoria por falta de espacio.

 Información del DataFrame Cacheado:
   • Particiones: 6
   • Storage Level: Disk Memory Deserialized 1x Replicated


### Paso 3.3: Diferencias entre .cache() y .persist()

Explorar los diferentes niveles de persistencia.

In [17]:
print("Comparando niveles de persistencia\n")
print("=" * 70)

# Crear DataFrame de prueba
df_test = df_taxi.filter(col("trip_distance") > 5).limit(50000)

# Test 1: MEMORY_ONLY (solo RAM, serializado)
print("\nMEMORY_ONLY")
print("   - Solo en RAM, serializado")
print("   - Si no cabe, las particiones se recomputan (no hay spill a disco)")
df_memory_only = df_test.persist(StorageLevel.MEMORY_ONLY)
start = time.time()
df_memory_only.count()
print(f"   Primera ejecución: {time.time()-start:.2f}s")
start = time.time()
df_memory_only.count()
print(f"   Segunda ejecución: {time.time()-start:.2f}s")
df_memory_only.unpersist()

# Test 2: MEMORY_AND_DISK (serializado en RAM + spill a disco)
print("\nMEMORY_AND_DISK (serializado en RAM, con spill a disco)")
print("   - Intenta RAM primero, luego disco")
print("   - Balance entre velocidad y confiabilidad")
df_memory_disk = df_test.persist(StorageLevel.MEMORY_AND_DISK)
start = time.time()
df_memory_disk.count()
print(f"   Primera ejecución: {time.time()-start:.2f}s")
start = time.time()
df_memory_disk.count()
print(f"   Segunda ejecución: {time.time()-start:.2f}s")
df_memory_disk.unpersist()

# Test 3: MEMORY_AND_DISK_DESER (equivalente de .cache())
print("\nMEMORY_AND_DISK_DESER (equivalente de .cache())")
print("   - Objetos deserializados en RAM, acceso más rápido")
print("   - Mayor uso de memoria que MEMORY_AND_DISK")
df_memory_deser = df_test.persist(StorageLevel.MEMORY_AND_DISK_DESER)
start = time.time()
df_memory_deser.count()
print(f"   Primera ejecución: {time.time()-start:.2f}s")
start = time.time()
df_memory_deser.count()
print(f"   Segunda ejecución: {time.time()-start:.2f}s")
df_memory_deser.unpersist()

print("\n" + "=" * 70)
print("\n Recomendaciones:")
print("   - Para análisis exploratorio o datasets pequeños que caben en RAM: MEMORY_ONLY")
print("   - Para uso general y datasets que pueden no caber en RAM: MEMORY_AND_DISK_DESER (equivalente a .cache())")
print("   - Para mayor ahorro de memoria (acepta serialización): MEMORY_AND_DISK")
print("   - En caso de duda: usar .cache() (simple y robusto)")

Comparando niveles de persistencia


MEMORY_ONLY
   - Solo en RAM, serializado
   - Si no cabe, las particiones se recomputan (no hay spill a disco)
   Primera ejecución: 1.07s
   Segunda ejecución: 0.12s

MEMORY_AND_DISK (serializado en RAM, con spill a disco)
   - Intenta RAM primero, luego disco
   - Balance entre velocidad y confiabilidad
   Primera ejecución: 0.95s
   Segunda ejecución: 0.09s

MEMORY_AND_DISK_DESER (equivalente de .cache())
   - Objetos deserializados en RAM, acceso más rápido
   - Mayor uso de memoria que MEMORY_AND_DISK
   Primera ejecución: 0.78s
   Segunda ejecución: 0.15s


 Recomendaciones:
   - Para análisis exploratorio o datasets pequeños que caben en RAM: MEMORY_ONLY
   - Para uso general y datasets que pueden no caber en RAM: MEMORY_AND_DISK_DESER (equivalente a .cache())
   - Para mayor ahorro de memoria (acepta serialización): MEMORY_AND_DISK
   - En caso de duda: usar .cache() (simple y robusto)


### Paso 3.4: Liberar Cache

Es importante liberar memoria cuando ya no se necesitan los datos cacheados.

In [18]:
print("Liberando cache...\n")

# Método 1: unpersist() en un DataFrame específico
df_cached.unpersist()
print("Cache liberado para df_cached")

# Método 2: Limpiar todo el cache del catálogo
spark.catalog.clearCache()
print("Todo el cache del catálogo limpiado")

print("\n Buenas prácticas:")
print("   • Liberar cache cuando se termine de usar un DataFrame")
print("   • En notebooks interactivos, limpiar cache periódicamente")
print("   • Monitorear Spark UI para ver uso de memoria")

Liberando cache...

Cache liberado para df_cached
Todo el cache del catálogo limpiado

 Buenas prácticas:
   • Liberar cache cuando se termine de usar un DataFrame
   • En notebooks interactivos, limpiar cache periódicamente
   • Monitorear Spark UI para ver uso de memoria


### Ejercicio 3: Decidir cuándo usar Cache

Para cada escenario, indicar si usaría cache (Si) o no (No) y explicar por qué.

**Escenario A:**
```python
df_filtered = df.filter(col("price") > 100)
df_filtered.write.parquet("output.parquet")
```

**Escenario B:**
```python
df_filtered = df.filter(col("price") > 100)
df_filtered.groupBy("category").count().show()
df_filtered.groupBy("region").agg(avg("price")).show()
df_filtered.groupBy("month").agg(sum("quantity")).show()
```

**Escenario C:**
```python
df_huge = spark.read.parquet("100TB_dataset.parquet")
df_processed = df_huge.filter(...).groupBy(...).agg(...)
df_processed.cache()
df_processed.count()
```

In [ ]:
# Escribir sus respuestas aquí


In [19]:
# Cerrar la SparkSession
spark.stop()
print("SparkSession cerrada correctamente.")

SparkSession cerrada correctamente.
